# Encrypted Traffic Classification with XGBoost

A presentation-ready walkthrough of how the PrivComm classifier turns network-flow statistics into traffic-class predictions. This notebook follows the repository's preprocessing and training pipeline, using the project dataset and a fixed random seed for reproducibility.

**Run from top to bottom.** The notebook cleans and engineers flow features, creates stratified train/validation/test splits, trains a multiclass XGBoost model, and reports results on the held-out test set. Generated model files are saved separately under `traffic-classifier/demo_artifacts/`; the application's existing model is not overwritten.

> The model learns from observable flow statistics (timing, packet rates, and byte rates). It classifies traffic patterns; it does not decrypt VPN payloads.

## 1. Set up the demo

The CSV is expected at the repository root as `consolidated_traffic_data.csv`. Install the traffic-classifier dependencies first if needed: `pip install -r traffic-classifier/requirements.txt`.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

SEED = 42
TARGET = "traffic_type"
candidate_paths = [
    Path.cwd() / "consolidated_traffic_data.csv",
    Path.cwd().parent / "consolidated_traffic_data.csv",
    Path.cwd().parent.parent / "consolidated_traffic_data.csv",
]
DATA_PATH = next((path.resolve() for path in candidate_paths if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find consolidated_traffic_data.csv. Place it in the repository root "
        "or update DATA_PATH in this setup cell."
    )

print(f"Dataset: {DATA_PATH}")
print(f"Reproducibility seed: {SEED}")

## 2. Load and inspect the raw flow data

Each row represents a network flow. `traffic_type` is the label to learn; the remaining columns describe flow behavior. First inspect the dimensions, columns, sample records, and class distribution.

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Raw dataset: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print(f"Target column present: {TARGET in raw_df.columns}")
display(raw_df.head())
display(raw_df.dtypes.rename("data type").to_frame())

if TARGET not in raw_df.columns:
    raise ValueError(f"Expected target column {TARGET!r} was not found in the CSV.")

class_counts = raw_df[TARGET].value_counts().sort_values(ascending=False)
ax = class_counts.plot(kind="bar", figsize=(12, 4), color="#2878B5")
ax.set(title="Raw samples per traffic class", xlabel="Traffic class", ylabel="Number of flows")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 3. Preprocess and engineer flow features

The preprocessing mirrors `traffic-classifier/src/preprocess.py`:

- Remove known identifier or leakage-prone columns, if present.
- Add five flow features: bytes per packet, forward/backward inter-arrival ratio, and log-scaled duration and rates.
- Drop rows with missing or infinite values and remove exact duplicates.

These engineered features are computed from the same flow statistics available to the classifier at inference time.

In [ ]:
df = raw_df.copy()
identifier_columns = [
    column for column in df.columns
    if column != TARGET and column.lower() in {
        "flow_id", "src_ip", "dst_ip", "src_port", "dst_port", "timestamp"
    }
]
if identifier_columns:
    print("Dropping identifier/leakage-risk columns:", identifier_columns)
    df = df.drop(columns=identifier_columns)
else:
    print("No known identifier/leakage-risk columns found.")

required_for_engineering = {
    "flowBytesPerSecond", "flowPktsPerSecond", "mean_fiat", "mean_biat",
    "duration"
}
missing_required = sorted(required_for_engineering.difference(df.columns))
if missing_required:
    raise ValueError(f"Cannot create the project's engineered features; missing: {missing_required}")

df["bytes_per_pkt"] = df["flowBytesPerSecond"] / (df["flowPktsPerSecond"] + 1e-5)
df["fiat_biat_ratio"] = df["mean_fiat"] / (df["mean_biat"] + 1e-5)
df["log_duration"] = np.log1p(np.maximum(0, df["duration"]))
df["log_bytes_sec"] = np.log1p(np.maximum(0, df["flowBytesPerSecond"]))
df["log_pkts_sec"] = np.log1p(np.maximum(0, df["flowPktsPerSecond"]))

before_cleaning = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
missing_rows = int(df.isna().any(axis=1).sum())
df = df.dropna()
rows_before_deduplication = len(df)
df = df.drop_duplicates().reset_index(drop=True)
duplicate_rows = rows_before_deduplication - len(df)
feature_columns = [column for column in df.columns if column != TARGET]

print(f"Rows before cleaning: {before_cleaning:,}")
print(f"Rows with missing/infinite values removed: {missing_rows:,}")
print(f"Exact duplicate rows removed: {duplicate_rows:,}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Predictor features: {len(feature_columns)}")
print("Engineered features: bytes_per_pkt, fiat_biat_ratio, log_duration, log_bytes_sec, log_pkts_sec")
display(df[feature_columns].describe().T.round(2))

## 4. Encode labels and create reproducible data splits

The string traffic labels are encoded as integers for model training. A stratified split preserves class proportions: **70% training**, **15% validation**, and **15% test**. The test set stays untouched until the final evaluation.

In [ ]:
label_encoder = LabelEncoder()
X = df[feature_columns].to_numpy(dtype=np.float64)
y = label_encoder.fit_transform(df[TARGET])
class_names = label_encoder.classes_.tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

split_summary = pd.DataFrame(
    {
        "samples": [len(y_train), len(y_val), len(y_test)],
        "share": [len(y_train) / len(y), len(y_val) / len(y), len(y_test) / len(y)],
    },
    index=["Train", "Validation", "Test"],
)
split_summary["share"] = split_summary["share"].map(lambda value: f"{value:.0%}")
display(split_summary)
print(f"Classes ({len(class_names)}): {', '.join(class_names)}")

train_class_share = pd.Series(y_train).value_counts(normalize=True).sort_index()
val_class_share = pd.Series(y_val).value_counts(normalize=True).sort_index()
split_plot = pd.DataFrame(
    {"Train": train_class_share, "Validation": val_class_share}, index=range(len(class_names))
)
split_plot.index = class_names
ax = split_plot.plot(kind="bar", figsize=(12, 4), color=["#2878B5", "#F28E2B"])
ax.set(title="Class proportions after stratified split", xlabel="Traffic class", ylabel="Share of split")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 5. Train the multiclass XGBoost model

XGBoost builds an ensemble of decision trees that progressively improves traffic-class predictions. Balanced sample weights help account for differences in class frequency. Validation loss is monitored for early stopping, so training can stop once validation performance stops improving.

In [ ]:
xgb_params = {
    "n_estimators": 400,
    "max_depth": 8,
    "learning_rate": 0.08,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "num_class": len(class_names),
    "eval_metric": "mlogloss",
    "random_state": SEED,
    "tree_method": "hist",
    "early_stopping_rounds": 20,
}
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

print("XGBoost configuration:")
display(pd.Series(xgb_params, name="value").to_frame())
print(f"Balanced training weights: {len(sample_weights):,} flows")

model = XGBClassifier(**xgb_params)
model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50,
)

best_iteration = getattr(model, "best_iteration", xgb_params["n_estimators"] - 1)
print(f"\nTraining complete. Best boosting iteration: {best_iteration}")

## 6. Visualize training progress

Compare multiclass log loss on the training and validation splits. A widening gap can indicate overfitting; early stopping selects the best validation iteration.

In [ ]:
training_history = model.evals_result()
train_loss = training_history["validation_0"]["mlogloss"]
val_loss = training_history["validation_1"]["mlogloss"]

plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Training", color="#2878B5")
plt.plot(val_loss, label="Validation", color="#F28E2B")
plt.axvline(best_iteration, linestyle="--", color="#555555", label=f"Best iteration ({best_iteration})")
plt.title("XGBoost training and validation loss")
plt.xlabel("Boosting round")
plt.ylabel("Multiclass log loss (lower is better)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Evaluate on the held-out test set

Only now do we use the test split. Accuracy summarizes all predictions; macro F1 gives each class equal weight, making it useful when class sizes differ. The confusion matrix shows which classes are most often confused.

In [ ]:
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")
test_weighted_f1 = f1_score(y_test, y_test_pred, average="weighted")
print(f"Held-out test flows: {len(y_test):,}")
print(f"Test accuracy: {test_accuracy:.3%}")
print(f"Test macro F1: {test_macro_f1:.3%}")
print(f"Test weighted F1: {test_weighted_f1:.3%}")
print("\nPer-class performance:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
    digits=3,
))

class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
    y_test, y_test_pred, labels=np.arange(len(class_names)), zero_division=0
)
macro_precision, macro_recall, _, _ = precision_recall_fscore_support(
    y_test, y_test_pred, average="macro", zero_division=0
)
weighted_precision, weighted_recall, _, _ = precision_recall_fscore_support(
    y_test, y_test_pred, average="weighted", zero_division=0
)
overall_metrics = pd.Series(
    {
        "Accuracy": test_accuracy,
        "Macro precision": macro_precision,
        "Macro recall": macro_recall,
        "Macro F1": test_macro_f1,
        "Weighted precision": weighted_precision,
        "Weighted recall": weighted_recall,
        "Weighted F1": test_weighted_f1,
    },
    name="Test score",
)
display(overall_metrics.map(lambda score: f"{score:.2%}").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
overall_metrics.plot(kind="bar", ax=axes[0], color="#2878B5", ylim=(0, 1))
axes[0].set(title="Overall held-out test metrics", ylabel="Score", xlabel="Metric")
axes[0].tick_params(axis="x", rotation=35)
class_metrics = pd.DataFrame(
    {"Precision": class_precision, "Recall": class_recall, "F1 score": class_f1},
    index=class_names,
)
class_metrics.plot(kind="bar", ax=axes[1], color=["#2878B5", "#F28E2B", "#59A14F"], ylim=(0, 1))
axes[1].set(title="Per-class held-out test metrics", ylabel="Score", xlabel="Traffic class")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(loc="lower right")
fig.tight_layout()
plt.show()

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(len(class_names)), normalize="true")
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    annot=len(class_names) <= 14,
    fmt=".0%",
    cbar_kws={"label": "Share of actual class"},
)
plt.title("Held-out test set: normalized confusion matrix")
plt.xlabel("Predicted traffic class")
plt.ylabel("Actual traffic class")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Which flow features mattered most?

This plot shows the model's relative feature-importance scores. It helps illustrate which statistical properties the trained trees relied on most; importance is not proof of causation.

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_columns).sort_values().tail(15)
ax = importance.plot(kind="barh", figsize=(9, 6), color="#2878B5")
ax.set(title="Top flow features by XGBoost importance", xlabel="Relative importance", ylabel="Feature")
plt.tight_layout()
plt.show()

## 9. Save the demo model separately

This optional final step saves the trained model, its label encoder, and a small metadata file under `traffic-classifier/demo_artifacts/`. These demo files are separate from the model artifacts used by the application.

In [ ]:
project_dir = DATA_PATH.parent / "traffic-classifier"
output_dir = project_dir / "demo_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "xgboost_traffic_classifier.json"
encoder_path = output_dir / "label_encoder.joblib"
metadata_path = output_dir / "demo_metadata.json"
model.save_model(model_path)
joblib.dump(label_encoder, encoder_path)

demo_metadata = {
    "model_name": "XGBoost Encrypted Traffic Classifier (demo retraining)",
    "dataset": DATA_PATH.name,
    "target_column": TARGET,
    "feature_columns": feature_columns,
    "classes": class_names,
    "random_seed": SEED,
    "best_iteration": int(best_iteration),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
}
metadata_path.write_text(json.dumps(demo_metadata, indent=2), encoding="utf-8")

print(f"Model saved: {model_path}")
print(f"Label encoder saved: {encoder_path}")
print(f"Demo metadata saved: {metadata_path}")

## Demo takeaway

The pipeline transforms packet-capture-derived flow statistics into engineered numerical features, learns patterns across labeled application and VPN traffic classes, and checks generalization on data held aside during training. Because inputs are traffic metadata and flow behavior, classification can work without decrypting the VPN payload.